# Stage 1 — MATLAB ↔ Python GPU Physics Parity

İlk hedef yalnızca şu deterministik bloğu Python/GPU'ya taşımaktır:

\[
(\text{array},a,d,f_c,K,\mu_{\rm XPR},\sigma_{\rm XPR})
\rightarrow
(\mu_H,\sigma_H^2,\bar d_T,\bar d_R)
\]

Bu aşamada amaç **önce doğruluk, sonra hızdır**.

- Parity modu: `float64 / complex128`
- Production modu: `float32 / complex64`
- GPU: PyTorch CUDA
- Gelecek dataset üretiminde bankalar aynı array şekline göre bucket edilip batch halinde GPU'ya verilecek.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, sys, json
import numpy as np
import torch
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')
PARITY_DIR = ROOT / 'python_parity_cases'
PARITY_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT       :", ROOT)
print("PARITY_DIR :", PARITY_DIR)
print("CUDA       :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU        :", torch.cuda.get_device_name(0))

## MATLAB tarafında önce golden case üret

MATLAB workspace'inde `gnb2ris`, `geometry`, `c0`, `lambda_0`, `K_BR`, `lsp_BR`
hazırken:

```matlab
export_no_cdl_parity_case( ...
    "gnb2ris_case.mat", ...
    gnb2ris, ...
    geometry.ris2gnb, ...
    geometry.gnb2ris, ...
    c0,lambda_0,K_BR,lsp_BR);
```

Aynısını `ris2ue` için de yap.

O `.mat` dosyalarını Drive'da:

`/content/drive/MyDrive/MyDrive/RIS/python_parity_cases/`

klasörüne koy.

In [ ]:
# Bu notebook ile birlikte verilen Python modülünü Drive'a koyduysan:
MODULE = ROOT / 'ris_gpu_physics_stage1.py'

# Alternatif: Colab'a doğrudan upload ettiysen /content altında olabilir.
if not MODULE.exists():
    alt = Path('/content/ris_gpu_physics_stage1.py')
    if alt.exists():
        MODULE = alt

assert MODULE.exists(), (
    "ris_gpu_physics_stage1.py bulunamadı. "
    "Dosyayı RIS root'a veya /content altına koy."
)

sys.path.insert(0, str(MODULE.parent))

from ris_gpu_physics_stage1 import (
    ArraySpec,
    compare_with_matlab_mat,
    benchmark_same_shape,
    generate_channel_moments_batch,
)

print("Loaded:", MODULE)

In [ ]:
# Golden case dosyalarını bul.
cases = sorted(PARITY_DIR.glob('*.mat'))
print("Cases:", [p.name for p in cases])
assert cases, "Önce MATLAB golden .mat case üret."

In [ ]:
# FLOAT64 / COMPLEX128 — doğruluk testi
rows = []

for p in cases:
    m = compare_with_matlab_mat(
        str(p),
        device='cuda' if torch.cuda.is_available() else 'cpu',
        parity=True,
    )
    rows.append({'case':p.name, **m})

parity_df = pd.DataFrame(rows)
display(parity_df)

# İlk hedefler:
# positions ve sigma2H machine precision civarı;
# muH de yaklaşık 1e-12 relative seviyesinde olmalı.
for _,r in parity_df.iterrows():
    assert r['dbarT_relFro'] < 1e-12
    assert r['dbarR_relFro'] < 1e-12
    assert r['sigma2H_rel'] < 1e-12
    assert r['muH_relFro'] < 1e-11

print("PASS: MATLAB ↔ Python Stage-1 parity")

In [ ]:
# PRODUCTION PRECISION kontrolü — float32 / complex64
rows = []

for p in cases:
    m = compare_with_matlab_mat(
        str(p),
        device='cuda' if torch.cuda.is_available() else 'cpu',
        parity=False,
    )
    rows.append({'case':p.name, **m})

prod_df = pd.DataFrame(rows)
display(prod_df)

print(
    "Bu tablo production float32/complex64 kullanıldığında "
    "MATLAB double referansından ne kadar uzaklaştığımızı gösterir."
)

In [ ]:
# GPU THROUGHPUT BENCHMARK
# Örnek: senin gösterdiğin gNB 4x2 dual-pol -> RIS 8x8 dual-pol.

tx = ArraySpec(4,2)
rx = ArraySpec(8,8)

if torch.cuda.is_available():
    gpu_bench = benchmark_same_shape(
        tx_spec=tx,
        rx_spec=rx,
        batch_size=4096,
        repeats=5,
        device='cuda',
    )
    print("GPU:", json.dumps(gpu_bench, indent=2))
else:
    print("CUDA yok; GPU benchmark atlandı.")

cpu_bench = benchmark_same_shape(
    tx_spec=tx,
    rx_spec=rx,
    batch_size=256,
    repeats=3,
    device='cpu',
)
print("CPU:", json.dumps(cpu_bench, indent=2))

## Bu test geçince sonraki aşama

Stage 2:

\[
\rho_{BR},\rho_{RU}
\]

hesaplarını MATLAB ile birebir eşleştirip GPU batch kernel'e taşıyacağız.

Sonra sırasıyla:

\[
UBR,\ URU,\ C,\ \mu_{\rm SNR},\sigma^2_{\rm Wick}
\]

ve V3 analytic feature'lar.

**XGB dataset generator'a ancak bu physics parity testleri geçtikten sonra başlayacağız.**